# TODO Softmax TensorFlow — Fashion-MNIST

Notebook này chia chương trình TensorFlow thành các cell có chú thích Markdown tiếng Việt. Cấu trúc tập trung vào các phần **import**, **TODO/task**, và **main training loop**, đồng thời giữ nguyên logic và các thiết lập trong code gốc.

> Chạy cell theo thứ tự. Cần có các module `util.py`, `logistic_np.py`, `softmax_np.py` và dữ liệu Fashion-MNIST trong môi trường hiện tại.

## 1. Import thư viện và cấu hình TensorFlow

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow.compat.v1 as tf
from util import get_mnist_data
from logistic_np import add_one
from softmax_np import *

tf.compat.v1.disable_eager_execution()

I0000 00:00:1791988046.250103   72682 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 2. Main — nạp dữ liệu và tiền xử lý

Nạp các tập train, validation và test; chuyển nhãn thành one-hot encoding; chuẩn hóa dữ liệu và thêm bias feature. Các hàm `create_one_hot`, `normalize` và `test` được import từ `softmax_np`.

In [2]:
np.random.seed(2020)
tf.set_random_seed(2020)

# Load data from file
# Make sure that fashion-mnist/*.gz files is in data/
train_x, train_y, val_x, val_y, test_x, test_y = get_mnist_data()
num_train = train_x.shape[0]
num_val = val_x.shape[0]
num_test = test_x.shape[0]

# generate_unit_testcase(train_x.copy(), train_y.copy())

# Convert label lists to one-hot (one-of-k) encoding
train_y = create_one_hot(train_y)
val_y = create_one_hot(val_y)
test_y = create_one_hot(test_y)

# Normalize data
train_x, val_x, test_x = normalize(train_x, val_x, test_x)

# Add bias feature
train_x = add_one(train_x)
val_x = add_one(val_x)
test_x = add_one(test_x)

Reading fashion MNIST data...
<class 'numpy.ndarray'>
Done reading


## 3. TODO 2.8

Phần triển khai hiện tại dùng `tf.constant` cho dữ liệu, để train nhanh hơn do constant được nạp trực tiếp vào VRAM của GPU thay vì cần feed mỗi epochs như placeholder. Trọng số `w` được khai báo là một `tf.Variable` kích thước `(785, 10)`, tương ứng 784 pixel cộng 1 bias và 10 lớp.

In [3]:
# [TODO 2.8] Create TF placeholders to feed train_x and train_y when training
train_x = tf.constant(train_x, dtype=tf.float32, name="image_pixel")
train_y = tf.constant(train_y, dtype=tf.float32, name="classifier_out")
test_x = tf.constant(test_x, dtype=tf.float32, name="test_pixel")
val_x = tf.constant(val_x, dtype=tf.float32, name="val_pixel")

# [TODO 2.8] Create weights (W) using TF variables
w = tf.Variable(
    tf.random.normal(shape=(785, 10), stddev=0.01),
    name="weight"
)

## 4. TODO 2.9

Tính logits (z) bằng phép nhân ma trận giữa dữ liệu và trọng số cho ba tập train, validation và test; sau đó áp dụng softmax để nhận xác suất dự đoán cho từng lớp.

In [4]:
# [TODO 2.9] Create a feed-forward operator
z = tf.matmul(train_x, w)
z_test = tf.matmul(test_x, w)
z_val = tf.matmul(val_x, w)

y_hat = tf.nn.softmax(z, name="probs")
y_hat_test = tf.nn.softmax(z_test, name="probs_test")
y_hat_val = tf.nn.softmax(z_val)

## 5. TODO 2.10

Dùng categorical cross-entropy cho train và validation. Xác suất được clip để tránh `log(0)`.

In [5]:
# [TODO 2.10] Write the cost function
eps = 1e-12

y_hat = tf.clip_by_value(y_hat, eps, 1.0)
y_hat_val = tf.clip_by_value(y_hat_val, eps, 1.0)

cost = -tf.reduce_mean(
    tf.reduce_sum(train_y * tf.math.log(y_hat), axis=1)
)

val_cost = -tf.reduce_mean(
    tf.reduce_sum(val_y * tf.math.log(y_hat_val), axis=1)
)

## 6. TODO 2.8 — SGD optimizer và thiết lập training

Khởi tạo Gradient Descent optimizer với learning rate được cung cấp trong code gốc. `train_op` là phép toán tối ưu hóa để cập nhật trọng số nhằm giảm training cost.

In [6]:
# Define hyperparameters and train-related parameters
num_epoch = 3347
learning_rate = 0.01

# [TODO 2.8] Create an SGD optimizer
optimizer = tf.train.GradientDescentOptimizer(learning_rate=learning_rate)
train_op = optimizer.minimize(cost)

# Some meta parameters
epochs_to_draw = 10
all_train_loss = []
all_val_loss = []
patience = 20
min_delta = 1e-6
best_val_loss = np.inf
wait = 0
plt.ion()
num_val_increase = 0

## 7. Main training loop — TODO 2.8 và TODO 2.11

Khởi tạo các biến TensorFlow rồi chạy session. Mỗi epoch lấy loss của train và validation, thực hiện cập nhật trọng số và lưu loss. TODO 2.11 được triển khai bằng early stopping: nếu validation loss không cải thiện ít nhất `min_delta` trong `patience` epoch liên tiếp thì dừng.

In [7]:
# Start training
init = tf.global_variables_initializer()

with tf.Session() as sess:
    sess.run(init)

    for e in range(num_epoch):
        # [TODO 2.8] Compute losses and update weights here
        train_loss = sess.run(cost)
        val_loss = sess.run(val_cost)

        # Update weights
        sess.run(train_op)

        all_train_loss.append(train_loss)
        all_val_loss.append(val_loss)

        # [TODO 2.11] Define your own stopping condition here
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(
                    f'Early stop at epoch {e}, '
                    f'best val loss = {best_val_loss:.4f}'
                )
                break

    y_hat_test = sess.run(y_hat_test)
    test(y_hat_test, test_y)

I0000 00:00:1791988062.162510   72682 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 760 MB memory:  -> device: 0, name: Quadro T1000, pci bus id: 0000:01:00.0, compute capability: 7.5
I0000 00:00:1791988062.186728   72682 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


Early stop at epoch 2857, best val loss = 0.5307
Confusion matrix:
[[0.91 0.   0.   0.04 0.   0.   0.05 0.   0.   0.  ]
 [0.   0.95 0.   0.05 0.   0.   0.   0.   0.   0.  ]
 [0.02 0.02 0.59 0.02 0.22 0.   0.13 0.   0.   0.  ]
 [0.07 0.03 0.   0.85 0.   0.   0.05 0.   0.   0.  ]
 [0.   0.   0.14 0.05 0.74 0.   0.07 0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.91 0.   0.08 0.02 0.  ]
 [0.23 0.   0.08 0.02 0.19 0.   0.47 0.   0.02 0.  ]
 [0.   0.   0.   0.   0.   0.04 0.   0.86 0.   0.1 ]
 [0.   0.   0.   0.02 0.   0.04 0.   0.02 0.93 0.  ]
 [0.   0.   0.   0.   0.   0.02 0.   0.   0.   0.98]]
Diagonal values:
[0.91 0.95 0.59 0.85 0.74 0.91 0.47 0.86 0.93 0.98]
